# 🛰️ Sentinel-AIOPs — Honest, Human-Gated AIOps
### A Grandmaster-style walkthrough: learned detection, deterministic localization, empirical validation

> **Thesis.** Detection may be *learned* on real public data. Localization and change correlation stay
> *deterministic*, inspectable, and *validated against ground truth*. The boundary is enforced in code and
> queryable at all times — no invented numbers, no cross-domain claims, no tuning to the test set.

**Repo:** `FrankAsanteVanLaarhoven/Sentinel-AIOPs` · **baseline:** `4af30ca`

---
### What this notebook does (end to end, on real public data)
| § | Layer | Dataset | We measure |
|---|---|---|---|
| 3 | Detection · **logs** | HDFS (`logfit-project/HDFS_v1`) | precision / recall / F1 / ROC-AUC |
| 4 | Detection · **metrics** | SMD (`NetManAIOps/OmniAnomaly`) | point-wise + point-adjusted F1 |
| 5 | Localization (deterministic) | synthetic scenarios | ground-truth agreement |
| 6 | Validation (empirical) | PetShop (`amazon-science`) | recall@1 / recall@3 / coverage |
| 7 | Consolidated results | — | one honest scoreboard |

**How to run.** Execute top-to-bottom from the repo's `notebooks/` folder with the ML extras installed
(`pip install -r ../engine/requirements-ml.txt`). Cells download *small subsets* for speed; the exact
committed numbers come from the `make` targets and are noted inline.


## 1 · Problem & hypotheses

An incident engine must **detect** an SLO breach, **localize** the causal service (not a symptom), and
**propose** a fix. The easy path learns all three end-to-end and destroys auditability. We instead test a
sequence of falsifiable hypotheses:

| # | Hypothesis | Verdict |
|---|---|---|
| **H1** | A 2-line rule *(root = elevated service with no elevated dependency)* localizes injected faults | ✅ 5/5 synthetic |
| **H2** | The *same* rule generalizes to real labelled incidents unchanged | ◑ PetShop r@1 0.265 / r@3 0.471 |
| **H3** | A tiny interpretable model on real logs gives a useful detection signal | ✅ HDFS F1 0.719 |
| **H4** | Learned detection extends to the metric modality | ◑ SMD F1 0.210 (honest, train-only) |
| **H5** | The learned/deterministic boundary is enforceable *and* queryable, zero side-effects | ✅ `/validation`, 16/16 tests |


In [ ]:
# --- environment ---------------------------------------------------------
# (Optional) install ML extras if running fresh:
# %pip install -q scikit-learn pandas pyarrow huggingface_hub joblib matplotlib

import os, sys, io, zipfile, urllib.request, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Make the shipped engine package importable so this notebook reuses the SAME
# code that runs in production (no reimplementation, no drift).
for cand in ["../engine/src", "engine/src", os.path.expanduser("~/Sentinel-AIOPs/engine/src")]:
    if os.path.isdir(cand):
        sys.path.insert(0, os.path.abspath(cand)); break

from sentinel.log_anomaly    import LogAnomalyDetector, event_template
from sentinel.metric_anomaly import MetricAnomalyDetector, point_adjust, prf
from sentinel.rca_petshop    import evaluate_dir, Z_DEFAULT
from sentinel.incident_agent  import detect, localize, causal_root
from sentinel.scenarios       import SCENARIO_META, build
from sentinel.telemetry_sim   import INC
from sentinel.tools           import TelemetryTools

plt.rcParams.update({"figure.facecolor":"white","axes.grid":True,"grid.alpha":.25,"font.size":10})
ACCENT = {"learned":"#7fa6d9","deterministic":"#5fd08a","empirical":"#e8b24c","crit":"#ff6a4d"}
DATA = "nb_data"; os.makedirs(DATA, exist_ok=True)
print("environment ready · reusing sentinel package from:", sys.path[0])


## 2 · Architecture — the three layers

Detection feeds *only* a scalar anomaly signal down. Localization is pure graph logic. Validation scores that
deterministic rule on real labelled incidents. `GET /validation` composes all three from committed cards.


In [ ]:
# A quick visual of the contract this whole system enforces.
fig, ax = plt.subplots(figsize=(9,3.4)); ax.axis("off")
layers = [("DETECTION  (learned · real public data)","Logs·HDFS  ·  Metrics·SMD",ACCENT["learned"]),
          ("LOCALIZATION  (deterministic · no weights)","causal_root: root = elevated w/ no elevated dep",ACCENT["deterministic"]),
          ("VALIDATION  (empirical · real corpora)","PetShop recall@1/@3 · coverage · failure modes",ACCENT["empirical"])]
for i,(t,s,c) in enumerate(layers):
    y = 2-i
    ax.add_patch(plt.Rectangle((0.05,y-0.34),0.9,0.62,facecolor=c,alpha=.18,edgecolor=c,lw=1.6))
    ax.text(0.5,y+0.05,t,ha="center",fontsize=11,weight="bold")
    ax.text(0.5,y-0.18,s,ha="center",fontsize=9,color="#444")
    if i<2: ax.annotate("",xy=(0.5,y-0.42),xytext=(0.5,y-0.34),arrowprops=dict(arrowstyle="-|>",color="#888"))
ax.set_xlim(0,1); ax.set_ylim(-0.5,2.6)
ax.set_title("Sentinel-AIOPs · learned detection ▸ deterministic localization ▸ empirical validation")
plt.tight_layout(); plt.show()


## 3 · Detection I — Logs (HDFS)

**Task.** Group raw HDFS log lines by `block_id` into sessions; classify each session as anomalous.
**Method.** Event-templating (mask volatile tokens) → bag-of-events counts → `LogisticRegression`.
Interpretable, fast, honest. *(1 shard here for speed; the committed card uses the same pipeline.)*


In [ ]:
from huggingface_hub import hf_hub_download
path = hf_hub_download("logfit-project/HDFS_v1", repo_type="dataset",
                       filename="data/train-00000-of-00005.parquet")
df = pd.read_parquet(path, columns=["content","block_id","anomaly"])
print(f"{len(df):,} raw log lines")
display(df.head(3))
print("\nexample event templates (volatile tokens masked):")
for s in df["content"].head(3): print("  ", event_template(s))


In [ ]:
# Templating -> vocabulary -> per-block bag-of-events sessions
df["tpl"] = df["content"].map(event_template)
top   = df["tpl"].value_counts().head(48).index
vocab = {t:i for i,t in enumerate(top)}; K = len(vocab)
df["tid"] = df["tpl"].map(lambda t: vocab.get(t, K))          # K = shared OTHER bucket
X = pd.crosstab(df["block_id"], df["tid"]).reindex(columns=range(K+1), fill_value=0)
y = df.groupby("block_id")["anomaly"].max().reindex(X.index).astype(int)
print(f"sessions={len(X):,} · anomaly rate={y.mean()*100:.2f}% · templates={df['tpl'].nunique()}")

fig,ax=plt.subplots(1,2,figsize=(10,3))
y.value_counts().sort_index().plot.bar(ax=ax[0],color=[ACCENT["deterministic"],ACCENT["crit"]])
ax[0].set_title("class balance (0=normal, 1=anomalous)"); ax[0].set_xticklabels(["normal","anomalous"],rotation=0)
df["tpl"].value_counts().head(12).plot.barh(ax=ax[1],color=ACCENT["learned"]); ax[1].invert_yaxis()
ax[1].set_title("top event templates"); plt.tight_layout(); plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

Xtr,Xte,ytr,yte = train_test_split(X.values, y.values, test_size=0.30,
                                   random_state=42, stratify=y.values)
det = LogAnomalyDetector(vocab_size=48).set_vocab(vocab)   # reuse the SHIPPED class
det.fit_features(Xtr, ytr)
proba = det.model.predict_proba(Xte)[:,1]; pred = (proba>=0.5).astype(int)

print(f"HELD-OUT (n={len(yte):,}, anomalies={int(yte.sum())})")
print(f"  precision {precision_score(yte,pred):.3f}")
print(f"  recall    {recall_score(yte,pred):.3f}")
print(f"  f1        {f1_score(yte,pred):.3f}")
print(f"  roc_auc   {roc_auc_score(yte,proba):.3f}")
print("  (committed card: P 0.992 / R 0.564 / F1 0.719 / AUC 0.787)")


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix
fpr,tpr,_ = roc_curve(yte,proba); pr,rc,_ = precision_recall_curve(yte,proba)
cm = confusion_matrix(yte,pred)
fig,ax=plt.subplots(1,3,figsize=(13,3.6))
ax[0].plot(fpr,tpr,color=ACCENT["learned"]); ax[0].plot([0,1],[0,1],"--",color="#bbb")
ax[0].set_title(f"ROC (AUC={roc_auc_score(yte,proba):.3f})"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
ax[1].plot(rc,pr,color=ACCENT["crit"]); ax[1].set_title("Precision–Recall"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
im=ax[2].imshow(cm,cmap="Blues"); ax[2].set_title("confusion matrix")
for (i,j),v in np.ndenumerate(cm): ax[2].text(j,i,f"{v:,}",ha="center",va="center")
ax[2].set_xticks([0,1]); ax[2].set_yticks([0,1]); ax[2].set_xlabel("pred"); ax[2].set_ylabel("true")
plt.tight_layout(); plt.show()


**Insight.** High precision, modest recall — a bag-of-events representation throws away intra-session
*order*, so it rarely false-alarms but misses ~⅓ of anomalies. Sequence models (DeepLog, LogLLM) score higher
on complete sessions. We report the honest envelope, not an upper bound.


## 4 · Detection II — Metrics (SMD)

**Task (unsupervised).** Learn "normal" from an anomaly-free training window; flag test points that no longer
fit. **Method.** Standardize → PCA (90% variance) → reconstruction error → smooth(5) → threshold at the
**99.9th percentile of *training* error** (never test labels). *(3 machines here for speed; the committed
card pools all 28 → F1 0.210.)*


In [ ]:
BASE = "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset"
def fetch(sub, m):
    p = os.path.join(DATA, f"{sub}_{m}.txt")
    if not os.path.exists(p): urllib.request.urlretrieve(f"{BASE}/{sub}/{m}.txt", p)
    return p

machines = ["machine-1-1","machine-2-1","machine-3-1"]
preds, labels = [], []
for m in machines:
    tr = np.loadtxt(fetch("train",m), delimiter=",")
    te = np.loadtxt(fetch("test", m), delimiter=",")
    lab= np.loadtxt(fetch("test_label",m)).astype(int)
    d  = MetricAnomalyDetector().fit(tr)          # reuse the SHIPPED class
    preds.append(d.predict(te)); labels.append(lab)
pred = np.concatenate(preds); label = np.concatenate(labels)
p,r,f = prf(pred,label); _,_,paf = prf(point_adjust(pred,label), label)
print(f"POOLED ({len(label):,} pts, {label.mean()*100:.1f}% anomalous, {len(machines)} machines)")
print(f"  precision {p:.3f} · recall {r:.3f} · F1 {f:.3f} · point-adjusted F1 {paf:.3f}")
print("  (committed card, all 28 machines: P 0.142 / R 0.403 / F1 0.210 / PA-F1 0.35)")


In [ ]:
# Reconstruction score vs ground-truth anomalies for one machine
m = "machine-1-1"
tr = np.loadtxt(fetch("train",m), delimiter=","); te = np.loadtxt(fetch("test",m), delimiter=",")
lab= np.loadtxt(fetch("test_label",m)).astype(int)
d  = MetricAnomalyDetector().fit(tr); s = d.score(te)
fig,ax=plt.subplots(figsize=(12,3))
ax.plot(s, color=ACCENT["learned"], lw=.7, label="reconstruction score")
ax.axhline(d.threshold, color=ACCENT["crit"], ls="--", lw=1, label="train-only threshold")
ax.fill_between(range(len(lab)), 0, s.max(), where=lab==1, color=ACCENT["crit"], alpha=.12, label="true anomaly")
ax.set_title(f"SMD {m}: PCA reconstruction score vs ground truth"); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()


**Insight — the honesty knob.** SMD papers report point-adjusted F1 with a *test-selected* threshold
(0.8–0.9). We set the threshold from **training only**, so even our PA-F1 (0.35) is far lower — and far more
trustworthy. The gap between these two protocols *is* the story.


## 5 · Localization — the deterministic core

The one function that runs live **and** in every evaluation below. Two lines. No weights.


In [ ]:
import inspect
print(inspect.getsource(causal_root))


In [ ]:
# H1: the SAME rule on 5 controlled scenarios (ground truth known)
rows=[]
for meta in SCENARIO_META:
    store = build(meta["id"]); tools = TelemetryTools(store)
    t,_ = detect(tools)
    root = localize(tools, t)[0] if t is not None else None
    gt   = store["ground_truth_root"]
    rows.append({"scenario":meta["id"],"localized":root,"ground_truth":gt,
                 "MTTD":(f"{t-INC}m" if t is not None else "-"),"correct":root==gt})
res = pd.DataFrame(rows)
display(res)
print(f"{res['correct'].sum()}/{len(res)} scenarios at {res['correct'].mean()*100:.0f}% ground-truth agreement")


## 6 · Validation on real incidents — PetShop

Now the honest test of **H2**: run the *unchanged* `causal_root` on 68 real labelled incidents. Only the
"elevated" signal is adapted (a fixed z≥3 on the target metric — disclosed, not tuned). recall@1 = the engine's
single answer.


In [ ]:
url = "https://github.com/amazon-science/petshop-root-cause-analysis/archive/refs/heads/main.zip"
zp  = os.path.join(DATA,"petshop")
if not os.path.isdir(zp):
    data = urllib.request.urlopen(url, timeout=90).read()
    zipfile.ZipFile(io.BytesIO(data)).extractall(zp)
ds = next(os.path.join(zp,d,"dataset") for d in os.listdir(zp)
          if os.path.isdir(os.path.join(zp,d,"dataset")))

ev = evaluate_dir(ds, z_thr=Z_DEFAULT, splits=("train","test"))
print(f"ALL: n={ev.n} · recall@1={ev.hit1/ev.n:.3f} · recall@3={ev.hit3/ev.n:.3f} · coverage={ev.detected/ev.n:.3f}")
for sc,s in ev.per_scenario.items():
    print(f"  {sc:18} n={s['n']:>2}  r@1={s['hit1']/s['n']:.3f}  r@3={s['hit3']/s['n']:.3f}")


In [ ]:
# Per-scenario recall + the tell-tale recall@3 >> recall@1 (granularity, not reasoning error)
labels_ = list(ev.per_scenario); r1=[ev.per_scenario[s]['hit1']/ev.per_scenario[s]['n'] for s in labels_]
r3=[ev.per_scenario[s]['hit3']/ev.per_scenario[s]['n'] for s in labels_]
x=np.arange(len(labels_)); w=.38
fig,ax=plt.subplots(figsize=(9,3.4))
ax.bar(x-w/2,r1,w,label="recall@1",color=ACCENT["crit"])
ax.bar(x+w/2,r3,w,label="recall@3",color=ACCENT["empirical"])
ax.axhline(ev.hit1/ev.n,ls="--",lw=1,color=ACCENT["crit"],alpha=.6)
ax.set_xticks(x); ax.set_xticklabels(labels_,rotation=15,ha="right"); ax.set_ylim(0,.7)
ax.set_title("PetShop localization — recall@1 vs recall@3 (all=0.265 / 0.471)"); ax.legend()
plt.tight_layout(); plt.show()
print("recall@3 ~ 2x recall@1 => the right REGION is found more often than the exact node (granularity).")
print(f"~{(1-ev.detected/ev.n)*100:.0f}% of incidents flag no node at all => a DETECTION-stage gap, not a localization error.")


**Insight — what the failures mean.** The ~29% detection gap and `recall@3 ≈ 2×recall@1` are the two
honest findings: some incidents never cross z≥3 on the target metric (a *detection* miss), and PetShop splits
one logical service into several nodes (so we flag a co-elevated sibling). Neither is a flaw in the causal
rule — which is exactly why we did **not** tune the rule to compensate.


### 6b · Closing the gap, within domain (the scoped increment)

The ~29% gap is a **detection** problem, so we fix it on PetShop's *own* signals: score each node on **all of
its metrics, two-sided** instead of the single target metric. Two variants: **broad** (any ≥1 metric deviates)
and **selective** (≥2 metrics — multivariate evidence). Same `causal_root`. We report **all / held-out test**,
because we picked the selective variant after comparing several — the test split keeps us honest.


In [ ]:
def _row(name, kind):
    a=evaluate_dir(ds, signal=kind, splits=("train","test")); t=evaluate_dir(ds, signal=kind, splits=("test",))
    return [name, a.hit1/a.n, t.hit1/t.n, a.hit3/a.n, t.hit3/t.n, a.detected/a.n, t.detected/t.n]
tbl = pd.DataFrame([_row("target (default)","target"), _row("within broad (>=1)","within_domain"),
                    _row("within selective (>=2)","within_domain_selective")],
    columns=["signal","r@1 all","r@1 test","r@3 all","r@3 test","cov all","cov test"])
display(tbl.round(3))
fig,ax=plt.subplots(figsize=(8.5,3)); x=np.arange(3); w=.26
for i,(m,c) in enumerate([("r@1 test",ACCENT["crit"]),("r@3 test",ACCENT["empirical"]),("cov test",ACCENT["deterministic"])]):
    ax.bar(x+(i-1)*w, tbl[m], w, label=m, color=c)
ax.set_xticks(x); ax.set_xticklabels(tbl["signal"],fontsize=8); ax.set_ylim(0,1.05); ax.legend(fontsize=8)
ax.set_title("Held-out test split: within-domain closes coverage; selectivity mitigates but doesn't restore precision")
plt.tight_layout(); plt.show()
print("Honest verdict: coverage gap closes; multivariate selectivity beats broad on recall@1, but on held-out")
print("data no within-domain signal restores full localization precision. The detection<->localization tension is real.")


## 7 · Consolidated scoreboard

One honest table. Modest where reality is hard; measured everywhere.


In [ ]:
summary = pd.DataFrame([
    ("Detection·Logs (HDFS)",    "F1",       0.719, "learned"),
    ("Detection·Metrics (SMD)",  "F1",       0.210, "learned"),
    ("Localization (synthetic)", "gt-agree", 1.000, "deterministic"),
    ("Validation·PetShop",       "recall@1", 0.265, "empirical"),
    ("Validation·PetShop",       "recall@3", 0.471, "empirical"),
], columns=["component","metric","value","layer"])
display(summary)
fig,ax=plt.subplots(figsize=(9,3.4))
ax.barh(summary["component"]+"  ("+summary["metric"]+")", summary["value"],
        color=[ACCENT[l] for l in summary["layer"]])
ax.set_xlim(0,1); ax.set_title("Sentinel-AIOPs — measured baseline (all real public data)")
for i,v in enumerate(summary["value"]): ax.text(v+.01,i,f"{v:.3f}",va="center",fontsize=9)
plt.tight_layout(); plt.show()


## 8 · What went wrong — and the mitigations (excerpt)

| Failure | Cause | Mitigation |
|---|---|---|
| Engine had no HTTP API | sample service only | wrote `engine_api.py` around tested functions |
| `/telemetry` 500 | `range` param shadowed the builtin | renamed `range_` (`alias="range"`) |
| ECharts blank in headless shots | entrance animation vs virtual-time | `animation:false` |
| Validation panel Detection empty | `/validation` fetch cached the pre-`detectors` shape | un-cached the fetch |
| SMD raw F1 ≈ 0.14 | fixed low threshold over-flags shifted test | fixed a-priori config; reported honestly |
| Peak-vs-mean, quantile sweeps tempting | would be test-tuning | tried, kept principled default (peak was worse), disclosed |
| A review claimed metrics that didn't exist | reviews a commit behind | reported the actual measured numbers, corrected the record |

*(Full log — 15 items — in `docs/MANUSCRIPT.md §10`.)*


## 9 · Decisions & honesty rails

- **Standalone detectors, not fused across domains** — HDFS ≠ SMD ≠ PetShop ≠ demo; we never claim
  cross-domain transfer we didn't measure (the 29% gap stays 29%).
- **Train-only thresholds; fixed a-priori configs** — no best-F1-on-test.
- **`causal_root` reused verbatim** in the evaluator (behavior-preserving extraction) — the rule the harness
  scores *is* the production rule; the green suite proves it's unchanged.
- **No invented numbers** — every figure here and in the served cards is a reproducible pipeline output.


## 10 · Conclusions & future work

**Conclusion.** A strict learned/deterministic boundary is not only enforceable but *legible*: two learned
detectors on real data, a deterministic core proven unchanged, and a measured real-incident baseline — all
queryable from `GET /validation` without running a pipeline. The numbers are modest where the problem is hard,
and we say so.

**Future work.**
1. **Within-domain detection** on PetShop's own metrics (reuse `MetricAnomalyDetector`), re-run
   `make validate-rca`, and report the *measured* change to the 29% coverage gap — the only honest way to move it.
2. Sequence-aware log detection (DeepLog / LogLLM) to lift recall.
3. Per-machine (still train-only) threshold calibration + more metric corpora (NAB, KPI).
4. More RCA corpora (RCAEval) for localization generalization.

```bash
make verify · make train-logdet · make train-metricdet · make validate-rca · make test
# GET /validation composes it all from committed cards.
```

*Built and measured honestly. Where results are modest, they are stated plainly; where a claim could not be
measured, it was not made.*
